# Feature B — Amazon Review Sentiment Analysis

**Project:** Amazon Marketplace Product Intelligence Platform  
**Feature:** B — Review Sentiment

## Objective
Classify each scraped Amazon review as **positive, neutral, or negative**, then create sentiment summaries by product and category.

This notebook uses **Gemini API to automatically generate the initial sentiment labels**, followed by label-quality checks, classical ML baselines, a transformer candidate, evaluation, error analysis, final prediction, and aggregation.

> **Important methodological limitation:** Gemini-generated labels are automatic/weak labels, not independently human-verified ground truth. Metrics against them measure agreement with the labeling process. The notebook does not present them as human-validated accuracy.

## Roadmap

1. Load and inspect the scraped dataset
2. Flatten the nested `reviews` JSON
3. Data-quality checks
4. Text preprocessing and EDA
5. Gemini API labeling
6. Confidence and consistency checks
7. Train/validation/test split
8. TF-IDF + Logistic Regression
9. TF-IDF + Linear SVM
10. Transformer classifier
11. Model comparison
12. Error analysis
13. Final model
14. Predict all reviews
15. Product/category sentiment summaries
16. Save artifacts and experiment report

In [ ]:
# Install once in a fresh environment if needed.
# !pip install -U pandas numpy matplotlib seaborn scikit-learn joblib tqdm google-genai
# !pip install -U transformers datasets accelerate torch

In [ ]:
import os
import re
import json
import html
import time
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 50)

print("Environment ready.")

In [ ]:
# Change this to your actual scraped CSV.
DATA_PATH = Path("data/your_scr.csv")

ARTIFACT_DIR = Path("artifacts/feature_b")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FLAT_REVIEWS_PATH = ARTIFACT_DIR / "flattened_reviews.csv"
CLEAN_REVIEWS_PATH = ARTIFACT_DIR / "clean_reviews.csv"
LABEL_CACHE_PATH = ARTIFACT_DIR / "gemini_labeled_reviews.csv"
LABELED_DATASET_PATH = ARTIFACT_DIR / "feature_b_gemini_labeled_dataset.csv"
FINAL_PREDICTIONS_PATH = ARTIFACT_DIR / "final_review_predictions.csv"
PRODUCT_SUMMARY_PATH = ARTIFACT_DIR / "product_sentiment_summary.csv"
CATEGORY_SUMMARY_PATH = ARTIFACT_DIR / "category_sentiment_summary.csv"
FINAL_MODEL_PATH = ARTIFACT_DIR / "final_sentiment_model.joblib"
REPORT_PATH = ARTIFACT_DIR / "feature_b_experiment_report.json"

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-2.5-flash")
GEMINI_BATCH_SIZE = 10
GEMINI_MAX_RETRIES = 5
GEMINI_RETRY_SECONDS = 5

TRANSFORMER_MODEL_NAME = os.getenv(
    "TRANSFORMER_MODEL_NAME",
    "distilbert-base-uncased"
)
MAX_LENGTH = 256
TRANSFORMER_EPOCHS = 2
TRANSFORMER_BATCH_SIZE = 16
TRANSFORMER_LEARNING_RATE = 2e-5

MIN_REVIEW_CHARS = 3

print("Dataset:", DATA_PATH)
print("Gemini:", GEMINI_MODEL)
print("Transformer:", TRANSFORMER_MODEL_NAME)

# 1. Load the scraped data

Expected input columns:

```text
category, productName, reviews
```

The `reviews` column contains a JSON array such as:

```json
[
  {"review": "I really like these earbuds."},
  {"review": "The battery is excellent."}
]
```

In [ ]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"{DATA_PATH} was not found. Update DATA_PATH in the configuration cell."
    )

df_raw = pd.read_csv(DATA_PATH)

required = {"category", "productName", "reviews"}
missing = required - set(df_raw.columns)

if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Shape:", df_raw.shape)
display(df_raw.head())

# 2. Flatten reviews

For sentiment classification we need **one row per review**, rather than one row per product.

The original product-level structure becomes:

`product_id | review_id | category | productName | review`

In [ ]:
def parse_reviews(value):
    if pd.isna(value):
        return []

    if isinstance(value, list):
        return value

    text = str(value).strip()

    if not text or text == "[]":
        return []

    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        try:
            parsed = json.loads(text.replace("\ufeff", "").strip())
        except Exception:
            return []

    return parsed if isinstance(parsed, list) else []


records = []

for product_idx, row in df_raw.iterrows():
    reviews = parse_reviews(row["reviews"])

    for review_idx, item in enumerate(reviews):
        if isinstance(item, dict):
            review_text = item.get("review", "")
        else:
            review_text = str(item)

        records.append({
            "product_id": product_idx,
            "review_id": f"{product_idx}_{review_idx}",
            "category": row["category"],
            "productName": row["productName"],
            "review": review_text,
        })

df_reviews = pd.DataFrame(records)

print("Flattened shape:", df_reviews.shape)
display(df_reviews.head(10))

In [ ]:
df_reviews.to_csv(FLAT_REVIEWS_PATH, index=False)

print("Products:", df_reviews["productName"].nunique())
print("Categories:", df_reviews["category"].nunique())
print("Reviews:", len(df_reviews))
print("\nMissing values:")
display(df_reviews.isna().sum().to_frame("missing"))

# 3. Data-quality analysis

Check:

- missing/empty reviews
- exact duplicate review text
- review length
- suspiciously short reviews
- product/category coverage

Exact duplicates are removed **only from the modeling dataset** to reduce train/test leakage. The original flattened dataset is retained for final product/category reporting.

In [ ]:
df_reviews["review"] = df_reviews["review"].fillna("").astype(str).str.strip()
df_reviews["review_chars"] = df_reviews["review"].str.len()
df_reviews["review_words"] = df_reviews["review"].str.split().str.len()

print("Empty reviews:", (df_reviews["review"] == "").sum())
print("Exact duplicate review texts:", df_reviews["review"].duplicated().sum())

display(df_reviews[["review_chars", "review_words"]].describe())

display(
    df_reviews.loc[
        df_reviews["review_chars"] <= 10,
        ["review_id", "productName", "review"]
    ].head(50)
)

# 4. Text preprocessing

Use light preprocessing for sentiment.

We intentionally do **not** remove stopwords aggressively because words such as `not`, `never`, and `no` can change sentiment.

Keep the original review and create a separate cleaned version.

In [ ]:
def clean_review(text):
    text = html.unescape(str(text))
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"https?://\S+|www\.\S+", " URL ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


df_model = df_reviews[
    df_reviews["review"].str.len() >= MIN_REVIEW_CHARS
].copy()

# Exact duplicate text can cause leakage across train/test.
df_model = (
    df_model
    .drop_duplicates(subset=["review"])
    .reset_index(drop=True)
)

df_model["review_clean"] = df_model["review"].map(clean_review)

print("Modeling rows:", len(df_model))

df_reviews.to_csv(FLAT_REVIEWS_PATH, index=False)
df_model.to_csv(CLEAN_REVIEWS_PATH, index=False)

display(df_model[["review", "review_clean"]].head(10))

# 5. Exploratory Data Analysis

In [ ]:
category_counts = df_model["category"].value_counts().head(20)

plt.figure(figsize=(12, 6))
category_counts.plot(kind="bar")
plt.title("Review Count by Category — Top 20")
plt.xlabel("Category")
plt.ylabel("Review Count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 5))
sns.histplot(df_model["review_words"], bins=50)
plt.title("Distribution of Review Length")
plt.xlabel("Words per Review")
plt.ylabel("Review Count")
plt.tight_layout()
plt.show()

# 6. Gemini API labeling

Gemini is used as the **automatic label generator**.

The model receives explicit definitions for the three classes and returns:

```json
[
  {
    "review_id": "1_0",
    "sentiment": "positive",
    "confidence": 0.95
  }
]
```

The labels are cached after each batch so the process can resume without re-labeling completed reviews.

In [ ]:
from google import genai

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    print("GEMINI_API_KEY is not set.")
    print("Set it before running the labeling function.")
else:
    client = genai.Client(api_key=GEMINI_API_KEY)
    print("Gemini client initialized.")

In [ ]:
LABEL_PROMPT = """
You are labeling Amazon customer reviews for a sentiment-analysis dataset.

Classify every review into exactly one of these classes.

POSITIVE:
The reviewer expresses an overall favorable opinion, satisfaction, praise,
recommendation, or clear approval of the product.

NEUTRAL:
The review is mainly factual, ambiguous, mixed without a clearly dominant
sentiment, or does not express a clearly positive or negative overall opinion.

NEGATIVE:
The reviewer expresses an overall unfavorable opinion, dissatisfaction,
criticism, disappointment, or clear problems with the product.

Rules:
- Judge the sentiment expressed by the review text.
- Do not classify only from product marketing language.
- Mixed reviews should be classified according to their overall sentiment.
- Very short or ambiguous reviews should generally be neutral with lower confidence.
- Do not add explanations outside the requested JSON.

Return ONLY valid JSON in this exact structure:

[
  {
    "review_id": "ID",
    "sentiment": "positive",
    "confidence": 0.95
  }
]

Allowed values:
positive, neutral, negative
"""


def make_batch_prompt(batch_df):
    items = [
        {
            "review_id": str(row["review_id"]),
            "review": row["review"]
        }
        for _, row in batch_df.iterrows()
    ]

    return LABEL_PROMPT + "\nReviews:\n" + json.dumps(
        items,
        ensure_ascii=False,
        indent=2
    )


def extract_json_array(text):
    text = text.strip()

    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
        text = re.sub(r"\s*```$", "", text)

    start = text.find("[")
    end = text.rfind("]")

    if start < 0 or end < 0:
        raise ValueError("No JSON array found in response.")

    return json.loads(text[start:end + 1])

In [ ]:
def label_batch_with_gemini(batch_df):
    if "client" not in globals():
        raise RuntimeError(
            "Gemini client is not initialized. Set GEMINI_API_KEY first."
        )

    prompt = make_batch_prompt(batch_df)

    for attempt in range(1, GEMINI_MAX_RETRIES + 1):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt
            )

            parsed = extract_json_array(response.text)

            expected_ids = set(batch_df["review_id"].astype(str))
            returned_ids = {str(x.get("review_id")) for x in parsed}

            if expected_ids != returned_ids:
                raise ValueError(
                    "Returned review IDs do not match the requested batch."
                )

            rows = []

            for item in parsed:
                sentiment = str(item.get("sentiment", "")).lower().strip()

                if sentiment not in {"positive", "neutral", "negative"}:
                    raise ValueError(
                        f"Invalid sentiment: {sentiment}"
                    )

                confidence = float(item.get("confidence", 0.0))
                confidence = min(1.0, max(0.0, confidence))

                rows.append({
                    "review_id": str(item["review_id"]),
                    "sentiment": sentiment,
                    "confidence": confidence,
                    "label_source": "gemini"
                })

            return pd.DataFrame(rows)

        except Exception as exc:
            print(
                f"Attempt {attempt}/{GEMINI_MAX_RETRIES} failed: {exc}"
            )

            if attempt == GEMINI_MAX_RETRIES:
                raise

            time.sleep(GEMINI_RETRY_SECONDS * attempt)


def run_gemini_labeling(input_df, batch_size=GEMINI_BATCH_SIZE):
    if LABEL_CACHE_PATH.exists():
        cached = pd.read_csv(LABEL_CACHE_PATH)
    else:
        cached = pd.DataFrame(
            columns=[
                "review_id",
                "sentiment",
                "confidence",
                "label_source"
            ]
        )

    labeled_ids = set(cached["review_id"].astype(str))

    remaining = input_df[
        ~input_df["review_id"].astype(str).isin(labeled_ids)
    ].copy()

    print("Total:", len(input_df))
    print("Already labeled:", len(labeled_ids))
    print("Remaining:", len(remaining))

    if remaining.empty:
        return cached

    all_results = [cached]

    batches = [
        remaining.iloc[i:i + batch_size]
        for i in range(0, len(remaining), batch_size)
    ]

    for batch_no, batch in enumerate(batches, 1):
        print(
            f"Batch {batch_no}/{len(batches)} — "
            f"{len(batch)} reviews"
        )

        batch_result = label_batch_with_gemini(batch)
        all_results.append(batch_result)

        current = pd.concat(all_results, ignore_index=True)
        current = current.drop_duplicates(
            subset=["review_id"],
            keep="last"
        )

        current.to_csv(LABEL_CACHE_PATH, index=False)

    return current


# RUN ONLY AFTER GEMINI_API_KEY IS SET:
# gemini_labels = run_gemini_labeling(df_model)

## Run labeling

The next cell is intentionally commented out. Uncomment it after setting `GEMINI_API_KEY`.

The process is resumable because every completed batch is saved to `gemini_labeled_reviews.csv`.

In [ ]:
# gemini_labels = run_gemini_labeling(df_model)

In [ ]:
if not LABEL_CACHE_PATH.exists():
    raise FileNotFoundError(
        f"{LABEL_CACHE_PATH} does not exist. "
        "Run the Gemini labeling cell first."
    )

gemini_labels = pd.read_csv(LABEL_CACHE_PATH)

df_labeled = df_model.merge(
    gemini_labels,
    on="review_id",
    how="inner",
    validate="one_to_one"
)

print("Labeled rows:", len(df_labeled))
display(df_labeled.head())

# 7. Analyze generated labels

Check:

- class balance
- Gemini confidence
- low-confidence reviews
- very short/ambiguous reviews

These are quality-control signals, not human ground truth.

In [ ]:
label_counts = df_labeled["sentiment"].value_counts()

display(
    label_counts.to_frame("count").assign(
        percentage=lambda x: x["count"] / len(df_labeled) * 100
    )
)

plt.figure(figsize=(8, 5))
label_counts.reindex(
    ["positive", "neutral", "negative"]
).plot(kind="bar")
plt.title("Gemini-Generated Sentiment Distribution")
plt.xlabel("Sentiment")
plt.ylabel("Review Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(df_labeled["confidence"], bins=20)
plt.title("Gemini Label Confidence")
plt.xlabel("Confidence")
plt.ylabel("Review Count")
plt.tight_layout()
plt.show()

display(df_labeled["confidence"].describe())

In [ ]:
df_labeled["needs_attention"] = (
    (df_labeled["confidence"] < 0.70)
    | (df_labeled["review_words"] <= 3)
)

print(
    "Reviews flagged for attention:",
    df_labeled["needs_attention"].sum()
)

display(
    df_labeled.loc[
        df_labeled["needs_attention"],
        ["review_id", "review", "sentiment", "confidence"]
    ].head(100)
)

# 8. Optional rating/text consistency check

If your scraped data contains star ratings, preserve them during flattening and use them as a **sanity-check signal**.

Do not use star ratings as the final sentiment labels, because the objective is to classify the sentiment expressed by the review text.

In [ ]:
# If your dataset has a rating field, add it to the flattening step
# and set RATING_COLUMN accordingly.

RATING_COLUMN = "rating"

if RATING_COLUMN in df_labeled.columns:
    def rating_to_expected_sentiment(rating):
        try:
            rating = float(rating)
        except Exception:
            return np.nan

        if rating >= 4:
            return "positive"
        if rating == 3:
            return "neutral"
        if rating <= 2:
            return "negative"

        return np.nan

    df_labeled["rating_expected_sentiment"] = (
        df_labeled[RATING_COLUMN]
        .map(rating_to_expected_sentiment)
    )

    mask = df_labeled["rating_expected_sentiment"].notna()

    print(
        "Rating/text consistency:",
        (
            df_labeled.loc[mask, "sentiment"]
            == df_labeled.loc[mask, "rating_expected_sentiment"]
        ).mean()
    )

    display(
        pd.crosstab(
            df_labeled["rating_expected_sentiment"],
            df_labeled["sentiment"],
            normalize="index"
        ).round(3)
    )
else:
    print("No rating column found. Skipping rating consistency analysis.")

# 9. Train / validation / test split

The split is stratified by the Gemini-generated labels.

**Interpretation of metrics:** model performance below is agreement with Gemini labels. It is not independent human-annotated accuracy.

In [ ]:
model_df = df_labeled[
    ["review_id", "review_clean", "sentiment"]
].dropna().copy()

train_df, temp_df = train_test_split(
    model_df,
    test_size=0.30,
    stratify=model_df["sentiment"],
    random_state=SEED
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["sentiment"],
    random_state=SEED
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

display(
    pd.DataFrame({
        "train": train_df["sentiment"].value_counts(normalize=True),
        "validation": val_df["sentiment"].value_counts(normalize=True),
        "test": test_df["sentiment"].value_counts(normalize=True),
    }).fillna(0)
)

# 10. Evaluation helpers

In [ ]:
LABEL_ORDER = ["negative", "neutral", "positive"]


def evaluate_predictions(y_true, y_pred, model_name):
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    p_weighted, r_weighted, f_weighted, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        )
    )

    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": p_macro,
        "recall_macro": r_macro,
        "f1_macro": f_macro,
        "precision_weighted": p_weighted,
        "recall_weighted": r_weighted,
        "f1_weighted": f_weighted,
    }


def show_evaluation(y_true, y_pred, name):
    print("=" * 80)
    print(name)
    print("=" * 80)

    print(
        classification_report(
            y_true,
            y_pred,
            labels=LABEL_ORDER,
            zero_division=0
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=LABEL_ORDER
    )

    fig, ax = plt.subplots(figsize=(6, 5))
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=LABEL_ORDER
    ).plot(ax=ax)

    ax.set_title(f"Confusion Matrix — {name}")
    plt.tight_layout()
    plt.show()

# 11. Baseline — TF-IDF + Logistic Regression

In [ ]:
tfidf_lr = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            random_state=SEED
        )
    )
])

tfidf_lr.fit(
    train_df["review_clean"],
    train_df["sentiment"]
)

val_pred_lr = tfidf_lr.predict(val_df["review_clean"])

show_evaluation(
    val_df["sentiment"],
    val_pred_lr,
    "TF-IDF + Logistic Regression"
)

results = [
    evaluate_predictions(
        val_df["sentiment"],
        val_pred_lr,
        "TF-IDF + Logistic Regression"
    )
]

# 12. Baseline — TF-IDF + Linear SVM

In [ ]:
tfidf_svm = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.95,
            sublinear_tf=True
        )
    ),
    (
        "classifier",
        LinearSVC(
            class_weight="balanced",
            random_state=SEED
        )
    )
])

tfidf_svm.fit(
    train_df["review_clean"],
    train_df["sentiment"]
)

val_pred_svm = tfidf_svm.predict(val_df["review_clean"])

show_evaluation(
    val_df["sentiment"],
    val_pred_svm,
    "TF-IDF + Linear SVM"
)

results.append(
    evaluate_predictions(
        val_df["sentiment"],
        val_pred_svm,
        "TF-IDF + Linear SVM"
    )
)

# 13. Transformer candidate

A pretrained encoder is included as a stronger neural candidate.

Default model:

`distilbert-base-uncased`

You can replace it with another suitable pretrained encoder through `TRANSFORMER_MODEL_NAME`.

In [ ]:
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)

label2id = {
    "negative": 0,
    "neutral": 1,
    "positive": 2,
}

id2label = {v: k for k, v in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(
    TRANSFORMER_MODEL_NAME
)

transformer_model = AutoModelForSequenceClassification.from_pretrained(
    TRANSFORMER_MODEL_NAME,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("Loaded:", TRANSFORMER_MODEL_NAME)

In [ ]:
def make_hf_dataset(frame):
    data = frame[["review_clean", "sentiment"]].copy()
    data["label"] = data["sentiment"].map(label2id)
    data = data.rename(columns={"review_clean": "text"})

    return Dataset.from_pandas(
        data[["text", "label"]],
        preserve_index=False
    )


train_hf = make_hf_dataset(train_df)
val_hf = make_hf_dataset(val_df)
test_hf = make_hf_dataset(test_df)


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH
    )


train_hf = train_hf.map(tokenize_batch, batched=True)
val_hf = val_hf.map(tokenize_batch, batched=True)
test_hf = test_hf.map(tokenize_batch, batched=True)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    p, r, f, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision_macro": p,
        "recall_macro": r,
        "f1_macro": f
    }


training_args = TrainingArguments(
    output_dir=str(ARTIFACT_DIR / "transformer_checkpoints"),
    learning_rate=TRANSFORMER_LEARNING_RATE,
    per_device_train_batch_size=TRANSFORMER_BATCH_SIZE,
    per_device_eval_batch_size=TRANSFORMER_BATCH_SIZE,
    num_train_epochs=TRANSFORMER_EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=transformer_model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    compute_metrics=compute_metrics,
)

# Uncomment to train:
# trainer.train()

In [ ]:
# Uncomment after transformer training.

# transformer_eval = trainer.evaluate(test_hf)
# print(transformer_eval)

# transformer_predictions = trainer.predict(test_hf).predictions.argmax(axis=-1)
# transformer_pred_labels = [
#     id2label[int(x)] for x in transformer_predictions
# ]
#
# show_evaluation(
#     test_df["sentiment"],
#     transformer_pred_labels,
#     "Transformer — Test"
# )

# 14. Compare the classical models

Macro F1 is emphasized because it gives equal importance to the three sentiment classes.

Add the transformer result here after training it.

In [ ]:
results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        "f1_macro",
        ascending=False
    ).reset_index(drop=True)
)

plt.figure(figsize=(10, 5))
sns.barplot(
    data=results_df,
    x="f1_macro",
    y="model"
)
plt.title("Model Comparison — Validation Macro F1")
plt.xlabel("Macro F1")
plt.ylabel("Model")
plt.xlim(0, 1)
plt.tight_layout()
plt.show()

# 15. Error analysis

Inspect incorrect predictions from the strongest classical model.

Useful error categories include:

- negation
- mixed sentiment
- sarcasm
- ambiguous language
- very short reviews
- spelling/grammar problems
- domain-specific terminology

In [ ]:
best_classical_name = (
    results_df
    .sort_values("f1_macro", ascending=False)
    .iloc[0]["model"]
)

if best_classical_name == "TF-IDF + Linear SVM":
    best_classical_model = tfidf_svm
else:
    best_classical_model = tfidf_lr

error_df = test_df.copy()
error_df["prediction"] = best_classical_model.predict(
    error_df["review_clean"]
)
error_df["correct"] = (
    error_df["sentiment"] == error_df["prediction"]
)

errors = error_df[~error_df["correct"]].copy()

print("Best classical model:", best_classical_name)
print("Errors:", len(errors), "/", len(test_df))

display(
    errors[
        ["review_id", "review_clean", "sentiment", "prediction"]
    ].head(100)
)

In [ ]:
# Neutral-related errors are often especially useful to inspect.

neutral_errors = errors[
    (errors["sentiment"] == "neutral")
    | (errors["prediction"] == "neutral")
]

display(
    neutral_errors[
        ["review_id", "review_clean", "sentiment", "prediction"]
    ].head(100)
)

# 16. Final model

For the default workflow, the strongest classical candidate is retrained on **all Gemini-labeled reviews**.

If your experiment shows that the transformer is the final model, save and use that transformer instead.

In [ ]:
if best_classical_name == "TF-IDF + Linear SVM":
    final_model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True
            )
        ),
        (
            "classifier",
            LinearSVC(
                class_weight="balanced",
                random_state=SEED
            )
        )
    ])
else:
    final_model = Pipeline([
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.95,
                sublinear_tf=True
            )
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=SEED
            )
        )
    ])

final_model.fit(
    model_df["review_clean"],
    model_df["sentiment"]
)

joblib.dump(final_model, FINAL_MODEL_PATH)

print("Saved:", FINAL_MODEL_PATH)

# 17. Predict sentiment for all scraped reviews

The final model is applied to the complete flattened review dataset.

This preserves all scraped review records for product/category analysis, while the training experiment used exact-text deduplication to reduce leakage.

In [ ]:
df_all_reviews = df_reviews.copy()

df_all_reviews["review_clean"] = (
    df_all_reviews["review"]
    .map(clean_review)
)

valid_mask = (
    df_all_reviews["review_clean"].str.len() >= MIN_REVIEW_CHARS
)

df_all_reviews["sentiment"] = "neutral"

df_all_reviews.loc[valid_mask, "sentiment"] = (
    final_model.predict(
        df_all_reviews.loc[
            valid_mask,
            "review_clean"
        ]
    )
)

display(
    df_all_reviews[
        [
            "review_id",
            "category",
            "productName",
            "review",
            "sentiment"
        ]
    ].head(20)
)

df_all_reviews.to_csv(
    FINAL_PREDICTIONS_PATH,
    index=False
)

print("Saved:", FINAL_PREDICTIONS_PATH)

# 18. Product-level sentiment summary

For each product:

- total reviews
- positive count/percentage
- neutral count/percentage
- negative count/percentage
- dominant sentiment

In [ ]:
product_counts = pd.crosstab(
    df_all_reviews["productName"],
    df_all_reviews["sentiment"]
)

for label in ["positive", "neutral", "negative"]:
    if label not in product_counts.columns:
        product_counts[label] = 0

product_summary = product_counts[
    ["positive", "neutral", "negative"]
].copy()

product_summary["total_reviews"] = product_summary.sum(axis=1)

for label in ["positive", "neutral", "negative"]:
    product_summary[f"{label}_pct"] = (
        product_summary[label]
        / product_summary["total_reviews"]
        * 100
    )

product_summary["dominant_sentiment"] = (
    product_summary[
        ["positive", "neutral", "negative"]
    ].idxmax(axis=1)
)

product_summary = (
    product_summary
    .reset_index()
    .rename(columns={"productName": "product_name"})
)

product_summary.to_csv(
    PRODUCT_SUMMARY_PATH,
    index=False
)

display(product_summary.head(20))

# 19. Category-level sentiment summary

In [ ]:
category_counts = pd.crosstab(
    df_all_reviews["category"],
    df_all_reviews["sentiment"]
)

for label in ["positive", "neutral", "negative"]:
    if label not in category_counts.columns:
        category_counts[label] = 0

category_summary = category_counts[
    ["positive", "neutral", "negative"]
].copy()

category_summary["total_reviews"] = category_summary.sum(axis=1)

for label in ["positive", "neutral", "negative"]:
    category_summary[f"{label}_pct"] = (
        category_summary[label]
        / category_summary["total_reviews"]
        * 100
    )

category_summary["dominant_sentiment"] = (
    category_summary[
        ["positive", "neutral", "negative"]
    ].idxmax(axis=1)
)

category_summary = category_summary.reset_index()

category_summary.to_csv(
    CATEGORY_SUMMARY_PATH,
    index=False
)

display(category_summary)

In [ ]:
category_plot = category_summary.set_index("category")[
    ["positive_pct", "neutral_pct", "negative_pct"]
]

category_plot.plot(
    kind="bar",
    figsize=(14, 7)
)

plt.title("Sentiment Distribution by Category")
plt.xlabel("Category")
plt.ylabel("Percentage of Reviews")
plt.xticks(rotation=60, ha="right")
plt.legend(title="Sentiment")
plt.tight_layout()
plt.show()

# 20. Save the labeled dataset

In [ ]:
df_labeled.to_csv(
    LABELED_DATASET_PATH,
    index=False
)

print("Saved:", LABELED_DATASET_PATH)

# 21. Experiment report

The report records the labeling method and the important limitation that the labels were generated automatically by Gemini.

In [ ]:
report = {
    "feature": "B - Review Sentiment",
    "classes": ["positive", "neutral", "negative"],
    "labeling_method": "Gemini API automatic labeling",
    "gemini_model": GEMINI_MODEL,
    "transformer_candidate": TRANSFORMER_MODEL_NAME,
    "random_seed": SEED,
    "total_scraped_reviews": len(df_reviews),
    "unique_modeling_reviews": len(model_df),
    "gemini_labeled_reviews": len(df_labeled),
    "train_reviews": len(train_df),
    "validation_reviews": len(val_df),
    "test_reviews": len(test_df),
    "final_classical_model": best_classical_name,
    "label_validation": [
        "Gemini confidence analysis",
        "short/ambiguous review inspection",
        "rating/text consistency when rating is available"
    ],
    "limitation": (
        "Gemini-generated labels are automatic/weak labels and are not "
        "independently human-verified ground truth. Metrics against them "
        "measure agreement with the labeling process."
    )
}

with open(REPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=4)

print(json.dumps(report, indent=4))
print("\nSaved:", REPORT_PATH)

# 22. Expected output structure

```text
artifacts/
└── feature_b/
    ├── flattened_reviews.csv
    ├── clean_reviews.csv
    ├── gemini_labeled_reviews.csv
    ├── feature_b_gemini_labeled_dataset.csv
    ├── final_review_predictions.csv
    ├── product_sentiment_summary.csv
    ├── category_sentiment_summary.csv
    ├── final_sentiment_model.joblib
    ├── feature_b_experiment_report.json
    └── transformer_checkpoints/        # if transformer is trained
```

## Streamlit integration

The application can load:

```python
import joblib

model = joblib.load(
    "artifacts/feature_b/final_sentiment_model.joblib"
)

sentiment = model.predict([review_text])[0]
```

For product/category dashboards, load the two summary CSV files.

## Final methodological note

Because this implementation uses Gemini to generate labels and does not manually annotate the dataset, the final report should explicitly describe the labels as **LLM-generated labels**. Do not claim that the resulting test accuracy/F1 is human-validated accuracy unless an independent human-annotated evaluation set is later created.